# 08 Figures and summary of the classification results

Draws the classification figures of the thesis from the result files of notebooks 02 to 07. Nothing is trained
here, so this notebook can be run again at any time. A figure whose result file is missing is skipped.

| Figure (file name) | Thesis | Result files |
|---|---|---|
| `res_zyla_zstack_flow` | Figure `fig:res-zyla-summary` | 02, 03, 04 |
| `10_input_size_Ph2` | Figure `fig:res-input-size-phantom` | 06 |
| `res_phantom_zstack_flow_apd` | Figure `fig:res-phantom-summary` | 05, 04, 07 |
| `app_confusion_*` (12 files) | Appendix, confusion matrices | 02, 03, 05, 07 |

Every figure is saved as PDF and PNG (300 dpi) in `results/figures/`, the caption as a `.txt` file next to it.

In [ ]:
%matplotlib inline
import pandas as pd
from IPython.display import display

import thesis_config as C
import thesis_style as S
import thesis_figures as TF

S.apply()
R, FIG = C.RESULTS_DIR, C.FIGURE_DIR
pd.set_option('display.precision', 3)


def metrics(name, scenario=None):
    # metrics of the CNN at 64 x 64, the chip network at 8 x 8 and the feature control
    return TF.select(TF.read(R, name), scenario)


def save(fig, name, caption):
    if fig is not None:
        S.save_figure(fig, name, caption, FIG)

## 1. Andor Zyla setup
Held-out stacks of the z-stack dataset (02), transfer from the z-stack to the Andor Zyla flow dataset (03) and
cross-validation within the Andor Zyla flow dataset (04).

In [ ]:
zyla = {'zs_zs': metrics('02_zstack', 'zstack_cv'),
        'zs_zf': metrics('03_zstack_to_zyla_flow'),
        'zf_zf': metrics('04_zyla_flow_cv')}
labels = {'zs_zs': 'Zyla z-stack\n→ Zyla z-stack', 'zs_zf': 'Zyla z-stack\n→ Zyla flow',
          'zf_zf': 'Zyla flow\n→ Zyla flow'}
if all(len(v) for v in zyla.values()):
    save(TF.comparison_figure(zyla, labels), 'res_zyla_zstack_flow',
         'Macro-F1 in the Andor Zyla setup for the CNN (64 × 64), the chip network (8 × 8) and the feature control. '
         'Left: z-stack, held-out stacks. Centre: trained on the z-stack, tested on the Andor Zyla flow dataset. '
         'Right: cross-validation within the Andor Zyla flow dataset, held-out recordings. Small points: single '
         'folds or seeds. Large markers: mean ± s.d.')

## 2. Input size (static Phantom, Ph2)

In [ ]:
size = TF.read(R, '06_input_size_ph2')
if len(size):
    save(TF.input_size_figure(size, C.PHANTOM_PX_UM), '10_input_size_Ph2',
         'Static Phantom Ph2 images: macro-F1 of the CNN and of the fully connected network against the input '
         'size N × N (N = 8: chip network). Mean ± s.d. over five folds and three seeds. Tick labels: input size and '
         'width of one input pixel (median crop side divided by N, limited to the camera pixel size).')

## 3. Phantom setup
Cross-validation within the static Phantom dataset (05), transfer from the static Phantom images to the Phantom flow
dataset (05), cross-validation within the Phantom flow dataset (04) and within the APD dataset, held-out videos (07).

In [ ]:
phantom = {'ps_ps': metrics('05_static_phantom_cv'),
           'ps_pf': metrics('05_static_phantom_to_flow'),
           'pf_pf': metrics('04_phantom_flow_cv'),
           'apd': metrics('07_apd_cv')}
labels = {'ps_ps': 'Phantom static\n→ Phantom static', 'ps_pf': 'Phantom static\n→ Phantom flow',
          'pf_pf': 'Phantom flow\n→ Phantom flow', 'apd': 'APD flow\n→ APD flow'}
if all(len(v) for v in phantom.values()):
    save(TF.comparison_figure(phantom, labels, separators=(3,)), 'res_phantom_zstack_flow_apd',
         'Macro-F1 in the Phantom setup for the CNN (64 × 64), the chip network (8 × 8) and the feature control: '
         'cross-validation within the static Phantom dataset, static Phantom images to the Phantom flow dataset, '
         'cross-validation within the Phantom flow dataset (held-out recordings) and within the APD dataset '
         '(held-out videos). Small points: single folds or seeds. Large markers: mean ± s.d.')

## 4. Confusion matrices (appendix)
Predictions of all folds, counts averaged over the three seeds, percentages normalized per true class (recall).

In [ ]:
RUNS = [  # file name, result file, scenario, title
    ('zstack065_stacks', '02_zstack', 'zstack_cv', 'z-stack 0.65 µm/px, held-out stacks'),
    ('zstack065_days', '02_zstack', 'zstack_cv_date', 'z-stack 0.65 µm/px, held-out days'),
    ('zyla065_transfer', '03_zstack_to_zyla_flow', None, 'z-stack → Andor Zyla flow, 0.65 µm/px'),
    ('phantom_{k}_cv', '05_static_phantom_cv', None, 'static Phantom, cross-validation (BF + Ph2)'),
    ('phantom_{k}_flow', '05_static_phantom_to_flow', None, 'static Phantom → Phantom flow'),
    ('apd_videos', '07_apd_cv', None, 'APD dataset, held-out videos'),
]
for key, source, scenario, title in RUNS:
    pred = TF.read(R, source, 'predictions')
    for kind in ['CNN', 'ChipNet']:
        p = TF.select(pred, scenario, kind)
        if not len(p):
            continue
        name = 'app_confusion_' + (key.format(k=kind.lower()) if '{k}' in key else f'{key}_{kind.lower()}')
        save(TF.confusion_figure(p, f'{TF.MODEL_SHORT[kind]}\n{title}'), name,
             f'Confusion matrix of the {TF.MODEL_SHORT[kind]}, {title}. Counts averaged over the seeds, all folds '
             'pooled. Percentages are normalized per true class (recall).')

## 5. Overview of all macro-F1 values
Mean, standard deviation and number of runs (folds × seeds) for every evaluation. The same values are given in the
tables of the Results chapter.

In [ ]:
SOURCES = ['02_zstack', '03_zstack_to_zyla_flow', '04_zyla_flow_cv', '04_phantom_flow_cv', '05_static_phantom_cv',
           '05_static_phantom_day', '05_static_phantom_to_flow', '07_apd_cv', '07_apd_day', '07_static_ph2_to_apd']
rows = [TF.read(R, s).assign(source=s) for s in SOURCES]
overview = pd.concat([r for r in rows if len(r)], ignore_index=True)
overview = (overview.groupby(['source', 'scenario', 'model'], sort=False).macro_f1
            .agg(['mean', 'std', 'min', 'max', 'count']).round(3))
display(overview)
overview.to_csv(R / 'overview_macro_f1.csv')